# Notebook 1 (Person 1): TripWorld KDD, Data Validation, Locality Features & HPC ETL

**CMPE 256 — Cross-City Travel Recommender System**

**What this notebook does (and rubric points it covers):**
1. **KDD & Data Understanding (15 pts):** Visualizes TripWorld's geographic distribution, trip length skew, Home → Travel category lift, and Signal vs. Noise POIs.
2. **Data Engineering Bonus (30 pts) & Data Validation (12 pts):** Runs the **Bronze → Silver → Gold** DuckDB pipeline (`ml/data_pipeline.py`) with **Pandera** schema validation and zero-leakage **User-Grouped 5-Fold CV**.
3. **Locality-Aware Feature Engineering (30 pts):** Builds city supply affordances, leakage-safe `tourist_ratio`, Bayesian Google rating, and multi-city **`opportunity_lift`** (what a traveler chose relative to what was actually available in their past cities).
4. **HPC ETL Benchmark (20 pts):** Benchmarks single-threaded Pandas vs. multi-threaded DuckDB on POI/trip aggregation and saves speedup charts to `reports/figures/`.
        

## 1. Colab / Local Environment Setup & Dataset Download

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/isaackimmi/cmpe-256-final-project.git"

if IN_COLAB:
    repo_dir = Path("/content/cmpe-256-final-project")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/requirements.txt", "matplotlib", "seaborn"], check=True)
else:
    # Local run: walk up to repo root if launched from notebooks/
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

Path("reports/figures").mkdir(parents=True, exist_ok=True)
print("Working directory:", Path.cwd(), "| Colab:", IN_COLAB)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

DATA_DIR = Path("data/tripworld")
if not (DATA_DIR / "travel_behaviors.parquet").exists():
    print("Downloading core TripWorld Parquet files (~421 MB)...")
    snapshot_download(
        repo_id="CRUISEResearchGroup/TripWorld",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        allow_patterns=[
            "MANIFEST.json",
            "metro_mapping_clean.json",
            "region_labels.parquet",
            "pois.parquet",
            "travel_behaviors.parquet",
            "metadata/metadata_all.parquet",
            "metadata/place_attributes.parquet",
        ],
    )
print("TripWorld core files ready:", sorted(p.name for p in DATA_DIR.glob("*.parquet")))

## 2. KDD Exploratory Data Analysis & Visualizations

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid", palette="deep")
con = duckdb.connect()
con.execute('''
    CREATE VIEW travel AS SELECT * FROM 'data/tripworld/travel_behaviors.parquet';
    CREATE VIEW pois AS SELECT * FROM 'data/tripworld/pois.parquet';
    CREATE VIEW meta AS SELECT * FROM 'data/tripworld/metadata/metadata_all.parquet';
    CREATE VIEW regions AS SELECT * FROM 'data/tripworld/region_labels.parquet';
''')

# Headline summary table
summary = con.sql('''
    SELECT
        (SELECT COUNT(*) FROM travel) AS cross_city_trips,
        (SELECT COUNT(DISTINCT user_id) FROM travel) AS unique_travelers,
        (SELECT COUNT(*) FROM pois) AS total_pois,
        (SELECT COUNT(*) FROM regions) AS metro_regions,
        (SELECT ROUND(100.0 * AVG(CAST(has_google_metadata AS INT)), 2) FROM pois) AS pct_google_meta,
        (SELECT ROUND(100.0 * AVG(CAST(has_reviews AS INT)), 2) FROM pois) AS pct_with_reviews
''').df()
display(summary)
        

In [ ]:
# Figure 1: Top Countries by Cross-City Trips & Trip Length Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_countries = con.sql('''
    SELECT r.country_name, COUNT(*) AS trips
    FROM travel t
    JOIN regions r ON t.r_o = r.region_id
    GROUP BY 1 ORDER BY trips DESC LIMIT 10
''').df()

sns.barplot(data=top_countries, y="country_name", x="trips", ax=axes[0], color="#3b82f6")
axes[0].set_title("Top 10 Destination Countries (Geographic Skew)")
axes[0].set_xlabel("Cross-City Trips")
axes[0].set_ylabel("")

trip_lens = con.sql('''
    SELECT
        CASE
            WHEN n_travel_ci = 2 THEN '2'
            WHEN n_travel_ci BETWEEN 3 AND 5 THEN '3-5'
            WHEN n_travel_ci BETWEEN 6 AND 10 THEN '6-10'
            WHEN n_travel_ci BETWEEN 11 AND 20 THEN '11-20'
            ELSE '21+'
        END AS bucket,
        COUNT(*) AS trips,
        MIN(n_travel_ci) AS sort_k
    FROM travel GROUP BY 1 ORDER BY sort_k
''').df()

sns.barplot(data=trip_lens, x="bucket", y="trips", ax=axes[1], color="#10b981")
axes[1].set_title("Destination Check-Ins per Trip Distribution")
axes[1].set_xlabel("Check-Ins in Destination City")
axes[1].set_ylabel("Number of Trips")

plt.tight_layout()
plt.savefig("reports/figures/01_kdd_geography_and_trip_lengths.png", dpi=150)
plt.show()
        

In [ ]:
# Figure 2: Home -> Travel Category Lift (What Surges on Trips) + Signal vs Noise Check-Ins
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lift_df = con.sql('''
    WITH h AS (
        SELECT u.venue_schema AS s, COUNT(*) AS cnt
        FROM (SELECT c_h FROM travel USING SAMPLE 50000 ROWS), UNNEST(c_h) AS _(u)
        WHERE u.venue_schema IS NOT NULL GROUP BY 1
    ),
    o AS (
        SELECT u.venue_schema AS s, COUNT(*) AS cnt
        FROM (SELECT c_o FROM travel USING SAMPLE 50000 ROWS), UNNEST(c_o) AS _(u)
        WHERE u.venue_schema IS NOT NULL GROUP BY 1
    ),
    tot AS (SELECT (SELECT SUM(cnt) FROM h) AS ht, (SELECT SUM(cnt) FROM o) AS ot)
    SELECT REPLACE(o.s, 'schema:', '') AS category,
           ROUND((o.cnt / tot.ot) / NULLIF(COALESCE(h.cnt, 1) / tot.ht, 0), 2) AS travel_lift
    FROM o LEFT JOIN h ON o.s = h.s CROSS JOIN tot
    WHERE o.cnt >= 800
    ORDER BY travel_lift DESC LIMIT 10
''').df()

sns.barplot(data=lift_df, y="category", x="travel_lift", ax=axes[0], color="#8b5cf6")
axes[0].axvline(1.0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Home → Travel Category Lift (Top 10)")
axes[0].set_xlabel("Lift Multiplier (Travel Share / Home Share)")
axes[0].set_ylabel("")

noise_df = con.sql('''
    SELECT
        CASE
            WHEN venue_schema IN ('schema:TrainStation','schema:SubwayStation','schema:BusStation','schema:Airport') THEN 'Transit / Airports (Noise)'
            WHEN venue_schema IN ('schema:House','schema:ApartmentComplex','schema:Residence') THEN 'Private Homes (Noise)'
            WHEN venue_schema IN ('schema:Restaurant','schema:FoodEstablishment','schema:CafeOrCoffeeShop','schema:BarOrPub') THEN 'Food & Drink (Signal)'
            WHEN venue_schema IN ('schema:Museum','schema:ArtGallery','schema:LandmarksOrHistoricalBuildings','schema:TouristAttraction','schema:Park','schema:Beach','schema:BuddhistTemple') THEN 'Culture & Attractions (Signal)'
            ELSE 'Other / Civic / Retail'
        END AS bucket,
        SUM(n_checkins) / 1e6 AS checkins_millions
    FROM pois GROUP BY 1 ORDER BY checkins_millions DESC
''').df()

palette = ["#ef4444" if "Noise" in b else "#2563eb" for b in noise_df["bucket"]]
sns.barplot(data=noise_df, y="bucket", x="checkins_millions", hue="bucket", legend=False, ax=axes[1], palette=palette)
axes[1].set_title("Why KDD Cleaning Matters: Signal vs. Noise Check-Ins")
axes[1].set_xlabel("Total Check-Ins (Millions)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("reports/figures/02_kdd_travel_lift_and_noise.png", dpi=150)
plt.show()
        

## 3. Build Shared Spine (Bronze → Silver → Gold + Pandera + 5-Fold CV)

In [ ]:
from ml.data_pipeline import build_pipeline
build_pipeline()
        

In [ ]:
# Run automated Pandera + Anti-Leakage test suite
import subprocess, sys
res = subprocess.run([sys.executable, "-m", "pytest", "-v", "tests/test_ml_spine.py"], capture_output=True, text=True)
print(res.stdout)
assert res.returncode == 0, res.stderr
        

## 4. Person 1: Locality-Aware & Opportunity-Normalized Features

In [ ]:
from ml.features.build_locality_features import build_locality_features
build_locality_features(val_fold=0)
        

In [ ]:
# Figure 3: Visualize Opportunity-Normalized Lift & Tourist-vs-Local Ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

poi_stats = con.sql('''
    SELECT REPLACE(venue_schema, 'schema:', '') AS category,
           tourist_ratio, bayesian_rating
    FROM 'data/gold/feat_poi_stats.parquet'
    WHERE venue_schema IN (
        'schema:Museum', 'schema:ArtGallery', 'schema:Beach',
        'schema:TouristAttraction', 'schema:Park', 'schema:Restaurant', 'schema:CafeOrCoffeeShop'
    )
      AND train_tourist_visits >= 3
''').df()

order = poi_stats.groupby("category")["tourist_ratio"].median().sort_values(ascending=False).index
sns.boxplot(data=poi_stats, y="category", x="tourist_ratio", order=order, ax=axes[0], showfliers=False)
axes[0].set_title("Tourist-vs.-Local Ratio by Category (Train Folds Only)")
axes[0].set_xlabel("Tourist Ratio (Higher = Iconic Visitor Destination)")
axes[0].set_ylabel("")

taste_sample = con.sql('''
    SELECT REPLACE(venue_schema, 'schema:', '') AS category,
           LEAST(opportunity_lift, 15.0) AS opportunity_lift_clipped
    FROM 'data/gold/feat_user_locality_taste.parquet'
    WHERE venue_schema IN ('schema:Museum', 'schema:Beach', 'schema:Park', 'schema:CafeOrCoffeeShop', 'schema:Restaurant')
    USING SAMPLE 50000 ROWS
''').df()

sns.violinplot(data=taste_sample, y="category", x="opportunity_lift_clipped", ax=axes[1], cut=0, inner="quartile")
axes[1].axvline(1.0, color="red", linestyle="--", label="Neutral (1.0x supply)")
axes[1].set_title("Opportunity-Normalized Taste Lift Distribution")
axes[1].set_xlabel("Opportunity Lift (User Share / City Supply Share, clipped at 15)")
axes[1].set_ylabel("")
axes[1].legend()

plt.tight_layout()
plt.savefig("reports/figures/03_locality_features_distribution.png", dpi=150)
plt.show()
        

## 5. HPC ETL Benchmark: Pandas vs. DuckDB Scaling (20 pts)

In [ ]:
import time
import pandas as pd

# Benchmark: Group 687K POIs by city + schema and compute supply shares + top-rated stats
t0 = time.perf_counter()
df_pd = pd.read_parquet("data/gold/pois_clean.parquet", columns=["poi_id", "region_id", "venue_schema", "google_rating", "n_checkins", "is_recommendable"])
df_pd = df_pd[df_pd["is_recommendable"]]
_ = df_pd.groupby(["region_id", "venue_schema"], as_index=False).agg(
    cnt=("poi_id", "count"),
    mean_rating=("google_rating", "mean"),
    sum_ci=("n_checkins", "sum"),
)
t_pandas = time.perf_counter() - t0

t0 = time.perf_counter()
_ = con.sql('''
    SELECT region_id, venue_schema, COUNT(*) AS cnt,
           AVG(google_rating) AS mean_rating, SUM(n_checkins) AS sum_ci
    FROM 'data/gold/pois_clean.parquet'
    WHERE is_recommendable = TRUE
    GROUP BY region_id, venue_schema
''').df()
t_duckdb = time.perf_counter() - t0

# Also benchmark exploded history aggregation (multi-million rows)
t0 = time.perf_counter()
h_pd = pd.read_parquet("data/gold/trip_history_exploded.parquet", columns=["trip_id", "venue_schema", "is_home"])
_ = h_pd.groupby(["trip_id", "venue_schema"]).size()
t_pandas_hist = time.perf_counter() - t0

t0 = time.perf_counter()
_ = con.sql('''
    SELECT trip_id, venue_schema, COUNT(*) AS cnt
    FROM 'data/gold/trip_history_exploded.parquet'
    GROUP BY trip_id, venue_schema
''').df()
t_duckdb_hist = time.perf_counter() - t0

bench = pd.DataFrame([
    {"Task": "POI Catalog Aggregation (687K rows)", "Engine": "Pandas (Single-Core)", "Seconds": t_pandas},
    {"Task": "POI Catalog Aggregation (687K rows)", "Engine": "DuckDB (Vectorized Multi-Core)", "Seconds": t_duckdb},
    {"Task": "Exploded Multi-City History (15M+ rows)", "Engine": "Pandas (Single-Core)", "Seconds": t_pandas_hist},
    {"Task": "Exploded Multi-City History (15M+ rows)", "Engine": "DuckDB (Vectorized Multi-Core)", "Seconds": t_duckdb_hist},
])
display(bench)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=bench, x="Task", y="Seconds", hue="Engine")
plt.title(f"HPC ETL Speedup: DuckDB is {t_pandas_hist/t_duckdb_hist:.1f}x Faster on Multi-City History")
plt.ylabel("Wall-Clock Time (seconds, lower is better)")
plt.tight_layout()
plt.savefig("reports/figures/04_hpc_etl_pandas_vs_duckdb.png", dpi=150)
plt.show()
        